In [1]:
import requests
import json
import numpy as np
import pandas as pd
import time
import warnings
import random
from collections import Counter as Counter

In [2]:
pd.set_option('display.max_rows', 5000)

# Collecting Fields Based on 2024 Variable Names

I identified the variable names based on their names in the 2024 ACS Detailed Tables Variables and 2024 ACS Data Profiles Variables tables (found [here](https://api.census.gov/data/2024/acs/acs5/variables.html) and [here](https://api.census.gov/data/2024/acs/acs5/profile/variables.html), respectively), using this notebook to validate the variable names

I later found that the variable names can differ from year to year. I updated this notebook to scrape the JSON versions of these tables across the various years, finding the variable names based on their Label in each table

In [64]:
demographic_fields = ['DP05_0002E', 'DP05_0003E' # total population by sex (male/female)
] + [f'DP05_000{str(i)}E' for i in range(5, 10) # total population by age
] + [f'DP05_00{str(i)}E' for i in range(10, 18) # total population by age, continued
] + [f'DP05_00{str(i)}E' for i in [35, 37, 45, 53, 61, 69, 74] # total population by race, without ethnicity
] + [f'DP05_00{str(i)}E' for i in [90] + list(range(96, 100)) # total population by race, hispanic or non-hispanic
] + [f'DP05_0{str(i)}E' for i in range(100, 103) # total population by race, hispanic or non-hispanic, continued
]

commuting_fields = ['B08301_003E', 'B08301_004E'
] + [f'B08301_0{i}E' for i in [10, 16, 17, 18, 19, 20, 21]
] + [f'B08006_0{i}E' for i in [20, 21, 25, 31, 32, 33, 34, 37, 38, 42, 48, 49, 50, 51]
]

commuting_by_age_fields = [f'B08101_0{i}E' for i in list(range(10, 17)) + list(range(18, 25)) + 
    list(range(26, 33)) + list(range(34, 41)) + list(range(42, 49)) + list(range(50, 57))]

commuting_by_race_fields = [f'B08105A_00{i}E' for i in range(2, 8)
] + [f'B08105B_00{i}E' for i in range(2, 8)
] + [f'B08105C_00{i}E' for i in range(2, 8)
] + [f'B08105D_00{i}E' for i in range(2, 8)
] + [f'B08105E_00{i}E' for i in range(2, 8)
] + [f'B08105F_00{i}E' for i in range(2, 8)
] + [f'B08105G_00{i}E' for i in range(2, 8)
] + [f'B08105I_00{i}E' for i in range(2, 8)
]

occupation_fields = [f'DP03_00{i}E' for i in range(27, 32)]
industry_fields = [f'DP03_00{i}E' for i in range(33, 46)]
worker_class_fields = [f'DP03_00{i}E' for i in range(47, 51)]

occupation_by_sex_and_race_fields = [f'C24010A_00{i}E' for i in range(2, 10)
] + [f'C24010A_0{i}E' for i in range(10, 14)
] + [f'C24010B_00{i}E' for i in range(2, 10)
] + [f'C24010B_0{i}E' for i in range(10, 14)
] + [f'C24010C_00{i}E' for i in range(2, 10) 
] + [f'C24010C_0{i}E' for i in range(10, 14) 
] + [f'C24010D_00{i}E' for i in range(2, 10) 
] + [f'C24010D_0{i}E' for i in range(10, 14) 
] + [f'C24010E_00{i}E' for i in range(2, 10) 
] + [f'C24010E_0{i}E' for i in range(10, 14) 
] + [f'C24010F_00{i}E' for i in range(2, 10) 
] + [f'C24010F_0{i}E' for i in range(10, 14) 
] + [f'C24010G_00{i}E' for i in range(2, 10) 
] + [f'C24010G_0{i}E' for i in range(10, 14) 
] + [f'C24010I_00{i}E' for i in range(2, 10) 
] + [f'C24010I_0{i}E' for i in range(10, 14) 
] 

# Validating the Column Names

In [65]:
demographic_fields_df = pd.read_csv('Data/Reference/US Census Bureau Data Documentation - Demographics.csv')
demographic_fields_df = demographic_fields_df[(demographic_fields_df.Name.str.contains('DP0'))]
commuting_fields_df = pd.concat([pd.read_csv('Data/Reference/US Census Bureau Data Documentation - Commuting.csv'), 
    pd.read_csv('Data/Reference/US Census Bureau Data Documentation - Commuting by Sex.csv')])
commuting_by_age_fields_df = pd.read_csv('Data/Reference/US Census Bureau Data Documentation - Commuting by Age.csv')
commuting_by_race_fields_df = pd.read_csv(
    'Data/Reference/US Census Bureau Data Documentation - Commuting by Race_Ethnicity.csv')

In [66]:
def clean_fields_df(fields_df):
    fields_df['LabelDepth'] = [label.count('!!') for label in fields_df.Label]

    fields_df['Metric'] = [label.split('!!')[0] for label in fields_df.Label]
    fields_df['DemographicLabel'] = [label.split('!!')[1] if label.count('!!') >= 1 else None
                                         for label in fields_df.Label]
    fields_df['SubLabel1'] = [label.split('!!')[2] if label.count('!!') >= 2 else 
                              None for label in fields_df.Label]
    fields_df['SubLabel2'] = [label.split('!!')[3] if label.count('!!') >= 3 else 
                              None for label in fields_df.Label]
    fields_df['SubLabel3'] = [label.split('!!')[4] if label.count('!!') >= 4 else 
                              None for label in fields_df.Label]
    fields_df['SubLabel4'] = [label.split('!!')[5] if label.count('!!') >= 5 else 
                              None for label in fields_df.Label]
    fields_df['FinalSubLabel'] = [label.split('!!')[label.count('!!')] for label in fields_df.Label]
    
    return(fields_df)

demographic_fields_df = clean_fields_df(demographic_fields_df)
commuting_fields_df = clean_fields_df(commuting_fields_df)
commuting_by_age_fields_df = clean_fields_df(commuting_by_age_fields_df)
commuting_by_race_fields_df = clean_fields_df(commuting_by_race_fields_df)

In [25]:
all_field_names_df = pd.concat([demographic_fields_df, commuting_fields_df, 
                                commuting_by_age_fields_df, commuting_by_race_fields_df])
all_field_names_df['FieldName'] = all_field_names_df.SubLabel3.str.cat(all_field_names_df.SubLabel4)

In [27]:
all_field_names_df.loc[(all_field_names_df.Name.isin(
    demographic_fields + commuting_fields + commuting_by_age_fields + commuting_by_race_fields
))][['Name', 'Label', 'SubLabel1', 'SubLabel2', 'SubLabel3', 'SubLabel4', 'FinalSubLabel']]

,Name,Label,SubLabel1,SubLabel2,SubLabel3,SubLabel4,FinalSubLabel
8,DP05_0002E,Estimate!!SEX AND AGE!!Total population!!Male,Total population,Male,None,None,Male
16,DP05_0003E,Estimate!!SEX AND AGE!!Total population!!Female,Total population,Female,None,None,Female
32,DP05_0005E,Estimate!!SEX AND AGE!!Total population!!Under...,Total population,Under 5 years,None,None,Under 5 years
40,DP05_0006E,Estimate!!SEX AND AGE!!Total population!!5 to ...,Total population,5 to 9 years,None,None,5 to 9 years
48,DP05_0007E,Estimate!!SEX AND AGE!!Total population!!10 to...,Total population,10 to 14 years,None,None,10 to 14 years
56,DP05_0008E,Estimate!!SEX AND AGE!!Total population!!15 to...,Total population,15 to 19 years,None,None,15 to 19 years
64,DP05_0009E,Estimate!!SEX AND AGE!!Total population!!20 to...,Total population,20 to 24 years,None,None,20 to 24 years
72,DP05_0010E,Estimate!!SEX AND AGE!!Total population!!25 to...,Total population,25 to 34 years,None,None,25 to 34 years
80,DP05_0011E,Estimate!!SEX AND AGE!!Total population!!35 to...,Total population,35 to 44 years,None,None,35 to 44 years
88,DP05_0012E,Estimate!!SEX AND AGE!!Total population!!45 to...,Total population,45 to 54 years,None,None,45 to 54 years


In [28]:
all_field_names_df.loc[(all_field_names_df.Name.isin(
    demographic_fields + commuting_fields + commuting_by_age_fields + commuting_by_race_fields
))].Name.unique().tolist()

['DP05_0002E',
 'DP05_0003E',
 'DP05_0005E',
 'DP05_0006E',
 'DP05_0007E',
 'DP05_0008E',
 'DP05_0009E',
 'DP05_0010E',
 'DP05_0011E',
 'DP05_0012E',
 'DP05_0013E',
 'DP05_0014E',
 'DP05_0015E',
 'DP05_0016E',
 'DP05_0017E',
 'DP05_0035E',
 'DP05_0037E',
 'DP05_0045E',
 'DP05_0053E',
 'DP05_0061E',
 'DP05_0069E',
 'DP05_0074E',
 'DP05_0090E',
 'DP05_0096E',
 'DP05_0097E',
 'DP05_0098E',
 'DP05_0099E',
 'DP05_0100E',
 'DP05_0101E',
 'DP05_0102E',
 'B08301_003E',
 'B08301_004E',
 'B08301_010E',
 'B08301_016E',
 'B08301_017E',
 'B08301_018E',
 'B08301_019E',
 'B08301_020E',
 'B08301_021E',
 'B08006_020E',
 'B08006_021E',
 'B08006_025E',
 'B08006_031E',
 'B08006_032E',
 'B08006_033E',
 'B08006_034E',
 'B08006_037E',
 'B08006_038E',
 'B08006_042E',
 'B08006_048E',
 'B08006_049E',
 'B08006_050E',
 'B08006_051E',
 'B08101_010E',
 'B08101_011E',
 'B08101_012E',
 'B08101_013E',
 'B08101_014E',
 'B08101_015E',
 'B08101_016E',
 'B08101_018E',
 'B08101_019E',
 'B08101_020E',
 'B08101_021E',
 'B081

# Scraping the JSONs to Collect the Variable Names for More Efficiently

In [29]:
year_start = 2011
year_end = 2025

field_name_df = pd.DataFrame()

table_type_extensions = {
    'detailed': '',
    'subject': '/subject',
    'profile': '/profile',
    'comparison': '/cprofile'
}     

def append_to_field_name_dict(year, table_type):
    url = f'https://api.census.gov/data/{year}/acs/acs5{table_type_extensions[table_type]}/variables.json'
    response = requests.get(url)
    
    try:
        fields_for_year_df = pd.DataFrame(response.json()['variables'].values())

        fields_for_year_df['Year'] = year
        fields_for_year_df['Name'] = response.json()['variables'].keys()
        
        global field_name_df
        field_name_df = pd.concat([field_name_df, fields_for_year_df])
        
    except:
        print(f'Error transcribing {table_type} table fields for {year}')
    time.sleep(5 + random.randint(-5000, 5000)/10000)
        
for year in range(year_start, year_end):
    append_to_field_name_dict(year, 'detailed')
    append_to_field_name_dict(year, 'subject')
    append_to_field_name_dict(year, 'profile')
    append_to_field_name_dict(year, 'comparison')
    
field_name_df['TableName'] = [field_name[0:field_name.index('_')] if '_' in field_name else None
                              for field_name in field_name_df.Name.values ]
field_name_df = field_name_df[['Year', 'Name', 'TableName'] + field_name_df.columns.tolist()[:-3]]
field_name_df = field_name_df.reset_index()

Error transcribing comparison table fields for 2011
Error transcribing comparison table fields for 2012
Error transcribing comparison table fields for 2013
Error transcribing comparison table fields for 2014


## Warning: Certain table concepts have different table names across the years
The following variable concepts_inconsistent_names contains a list of table concepts whose table names change at least once from 2011-2024. The code for collecting the field names for each year does not account for changes to table names, so special consideration needs to be taken if using these fields

In [30]:
# table_names_per_concept = field_name_df[~(field_name_df.TableName.isna())]\
#     .groupby(['TableName', 'concept'], as_index=False).count()\
#     [['TableName', 'concept']].groupby('concept', as_index=False).count()

# concepts_inconsistent_names = table_names_per_concept.loc[(table_names_per_concept.TableName > 1)]\
#     .sort_values('TableName', ascending=False).concept.values.tolist()

# Cleaning Up the Label Names

In [67]:
def standardize_label(label):
    standardized_label = label.replace('Total population!!', '')\
        .replace(':', '')\
        .replace('Total!!', '')\
        .replace('Taxicab', 'Taxi or ride-hailing services')\
        .replace('Public transportation (excluding taxicab)', 'Public transportation')\
        .replace('Public transportation:', 'Public transportation')\
        .replace('Worked from home', 'Worked at home')\
        .replace('CLASS OF WORKER!!Civilian employed population 16 years and over!!', 'CLASS OF WORKER!!')\
        .replace('OCCUPATION!!Civilian employed population 16 years and over!!', 'OCCUPATION!!')\
        .replace('INDUSTRY!!Civilian employed population 16 years and over!!', 'INDUSTRY!!')\
        .lower()
    return(standardized_label)

In [68]:
field_names_2024 = demographic_fields + commuting_fields + \
    commuting_by_age_fields + commuting_by_race_fields + \
    occupation_fields + industry_fields + \
    worker_class_fields + occupation_by_sex_and_race_fields

table_names_2024 = list(set([field_name[0:field_name.index('_')] for field_name in field_names_2024]))

field_name_df['StandardizedLabel'] = [standardize_label(label) for label in field_name_df.label.values]
field_labels_2024_df = field_name_df.loc[(field_name_df.Name.isin(field_names_2024)) & 
                      (field_name_df.Year == 2024)][['StandardizedLabel', 'TableName', 'Name']]

In [69]:
field_labels_all = field_labels_2024_df[['TableName', 'StandardizedLabel']]\
    .merge(field_name_df.loc[(field_name_df.Year != 2024)][['Year', 'TableName', 'StandardizedLabel']], 
           on=['TableName', 'StandardizedLabel'], 
           how='outer', indicator=True)

field_labels_all['LabelParts'] = [label.split('!!') for label in field_labels_all.StandardizedLabel]

field_labels_missing = field_labels_all.loc[(field_labels_all._merge == 'left_only')]\
    [['TableName', 'StandardizedLabel']]
field_labels_remaining = field_labels_all.loc[(field_labels_all._merge == 'right_only')]\
    .loc[(field_labels_all.TableName).isin(field_labels_2024_df.TableName.values)]\
    [['TableName', 'StandardizedLabel']]

In [70]:
duplicate_field_labels = field_name_df[['TableName', 'StandardizedLabel', 'Name', 'Year']]\
    .merge(field_labels_2024_df.drop('Name', axis=1), on=['TableName', 'StandardizedLabel'])\
    .groupby(['Year', 'TableName', 'StandardizedLabel'], as_index=False).count()
duplicate_field_labels = duplicate_field_labels.loc[(duplicate_field_labels.Name) > 1]\
    .rename(columns={'Name': 'Count'})

In [71]:
duplicate_field_labels.head()

,Year,TableName,StandardizedLabel,Count
245,2011,DP05,estimate!!race!!two or more races,2
258,2011,DP05,estimate!!sex and age!!female,3
259,2011,DP05,estimate!!sex and age!!male,3
506,2012,DP05,estimate!!race!!two or more races,2
519,2012,DP05,estimate!!sex and age!!female,3


In [72]:
checking_duplicate_fields = duplicate_field_labels\
    .merge(field_name_df, on=['Year', 'TableName', 'StandardizedLabel'])\
    [['Year', 'TableName', 'StandardizedLabel', 'label', 'Name']]\
    .groupby(['Year', 'TableName', 'StandardizedLabel', 'label'], as_index=False).count()\
    .groupby(['Year', 'TableName', 'StandardizedLabel'], as_index=False).count()\
    .rename(columns={'label': 'Count'})

In [73]:
checking_duplicate_fields.loc[checking_duplicate_fields.Count > 1]

,Year,TableName,StandardizedLabel,Count,Name
6,2013,DP05,estimate!!race!!two or more races,2,2
7,2014,DP05,estimate!!race!!two or more races,2,2
8,2015,DP05,estimate!!race!!two or more races,2,2
9,2016,DP05,estimate!!race!!two or more races,2,2


In [74]:
duplicate_field_labels.merge(field_name_df, on=['Year', 'TableName', 'StandardizedLabel'])\
    [['Year', 'TableName', 'StandardizedLabel', 'Name', 'label']]

,Year,TableName,StandardizedLabel,Name,label
0,2011,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Two or more races
1,2011,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races
2,2011,DP05,estimate!!sex and age!!female,DP05_0024E,Estimate!!SEX AND AGE!!Female
3,2011,DP05,estimate!!sex and age!!female,DP05_0027E,Estimate!!SEX AND AGE!!Female
4,2011,DP05,estimate!!sex and age!!female,DP05_0003E,Estimate!!SEX AND AGE!!Female
5,2011,DP05,estimate!!sex and age!!male,DP05_0026E,Estimate!!SEX AND AGE!!Male
6,2011,DP05,estimate!!sex and age!!male,DP05_0023E,Estimate!!SEX AND AGE!!Male
7,2011,DP05,estimate!!sex and age!!male,DP05_0002E,Estimate!!SEX AND AGE!!Male
8,2012,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Two or more races
9,2012,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races


Most of the fields with duplicated StandardizedLabels also have identical labels, so they should return the same data and we will only keep the first of these duplicate fields in each year.

For each of the years from 2013 to 2016, there are two rows in table DP05 that have similar labels. These labels don't have any apparent difference in meaning, and some quick testing indicates they consistently return the same data, so we will just keep one of the column names

In [75]:
duplicate_field_labels[(duplicate_field_labels.TableName == 'DP05') &
                      (duplicate_field_labels.Year.isin([2013, 2014, 2015, 2016]))]\
    .merge(field_name_df, on=['Year', 'TableName', 'StandardizedLabel'])\
    [['Year', 'TableName', 'StandardizedLabel', 'Name', 'label']]

,Year,TableName,StandardizedLabel,Name,label
0,2013,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Total population!!Two or more ...
1,2013,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races
2,2014,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Total population!!Two or more ...
3,2014,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races
4,2015,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Total population!!Two or more ...
5,2015,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races
6,2016,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Total population!!Two or more ...
7,2016,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races


# Identifying the Necessary Columns for Each Year

In [76]:
field_names_by_year_df = field_labels_2024_df[['TableName', 'StandardizedLabel', 'Name']]\
    .rename(columns={'Name':'FieldName2024'})\
    .merge(field_name_df[['Year','TableName', 'StandardizedLabel', 'Name']],
          on=['TableName', 'StandardizedLabel'])\
    .sort_values('Name')\
    .groupby(['Year', 'TableName', 'StandardizedLabel'], as_index=False)\
    .first()

In [77]:
field_names_by_year_df.groupby(['Year', 'Name'], as_index=False).count()\
    .groupby(['Year']).count()

,Name,TableName,StandardizedLabel,FieldName2024
Year,,,,
2011,261,261,261,261
2012,261,261,261,261
2013,261,261,261,261
2014,261,261,261,261
2015,261,261,261,261
2016,261,261,261,261
2017,261,261,261,261
2018,261,261,261,261
2019,261,261,261,261


In [78]:
field_names_by_year_df.loc[(field_names_by_year_df.Year == 2024) &
                          ~(field_names_by_year_df.StandardizedLabel.isin(
                          field_names_by_year_df.loc[(field_names_by_year_df.Year == 2011)].StandardizedLabel.values))]\
    .StandardizedLabel.values

array([], dtype=object)

In [79]:
field_names_by_year_df

,Year,TableName,StandardizedLabel,FieldName2024,Name
0,2011,B08006,estimate!!female!!bicycle,B08006_048E,B08006_048E
1,2011,B08006,"estimate!!female!!car, truck, or van!!carpooled",B08006_038E,B08006_038E
2,2011,B08006,"estimate!!female!!car, truck, or van!!drove alone",B08006_037E,B08006_037E
3,2011,B08006,estimate!!female!!public transportation,B08006_042E,B08006_042E
4,2011,B08006,estimate!!female!!taxi or ride-hailing service...,B08006_050E,B08006_050E
5,2011,B08006,estimate!!female!!walked,B08006_049E,B08006_049E
6,2011,B08006,estimate!!female!!worked at home,B08006_051E,B08006_051E
7,2011,B08006,estimate!!male!!bicycle,B08006_031E,B08006_031E
8,2011,B08006,"estimate!!male!!car, truck, or van!!carpooled",B08006_021E,B08006_021E
9,2011,B08006,"estimate!!male!!car, truck, or van!!drove alone",B08006_020E,B08006_020E


In [80]:
field_rename_dict = {
    'zip code tabulation area': 'ZIPCode',
    'DP05_0002E': 'MalePopulation',
    'DP05_0003E': 'FemalePopulation',
    'DP05_0005E': 'AgeUnder5',
    'DP05_0006E': 'Age5to9',
    'DP05_0007E': 'Age10to14',
    'DP05_0008E': 'Age15to19',
    'DP05_0009E': 'Age20to24',
    'DP05_0010E': 'Age25to34',
    'DP05_0011E': 'Age35to44',
    'DP05_0012E': 'Age45to54',
    'DP05_0013E': 'Age55to59',
    'DP05_0014E': 'Age60to64',
    'DP05_0015E': 'Age65to74',
    'DP05_0016E': 'Age75to84',
    'DP05_0017E': 'Age85AndOver',
    'DP05_0035E': 'TwoOrMoreRacesPopulation',
    'DP05_0037E': 'WhitePopulation',
    'DP05_0045E': 'BlackPopulation',
    'DP05_0053E': 'IndigenousPopulation',
    'DP05_0061E': 'AsianPopulation',
    'DP05_0069E': 'PacificIslanderPopulation',
    'DP05_0074E': 'OtherRacePopulation',
    'DP05_0090E': 'HispanicPopulation',
    'DP05_0096E': 'WhiteNonHispanicPopulation',
    'DP05_0097E': 'BlackNonHispanicPopulation',
    'DP05_0098E': 'IndigenousNonHispanicPopulation',
    'DP05_0099E': 'AsianNonHispanicPopulation',
    'DP05_0100E': 'PacificIslanderNonHispanicPopulation',
    'DP05_0101E': 'OtherRaceNonHispanicPopulation',
    'DP05_0102E': 'TwoOrMoreRacesNonHispanicPopulation',
    'DP03_0027E': 'WorkersManagementBusinesScienceAndArts',
    'DP03_0028E': 'WorkersService',
    'DP03_0029E': 'WorkersSalesAndOffice',
    'DP03_0030E': 'WorkersResourcesConstructionAndMaintenance',
    'DP03_0031E': 'WorkersProductionTransportAndMaterialMoving',
    'DP03_0033E': 'WorkersAgricultureForestryFishingHuntingMiningIndustry',
    'DP03_0034E': 'WorkersConstructionIndustry',
    'DP03_0035E': 'WorkersManufacturingIndustry',
    'DP03_0036E': 'WorkersWholesaleTradeIndustry',
    'DP03_0037E': 'WorkersRetailTradeIndustry',
    'DP03_0038E': 'WorkersTransportationWarehousingAndUtilitiesIndustry',
    'DP03_0039E': 'WorkersInformationIndustry',
    'DP03_0040E': 'WorkersFinanceInsuranceRealEstateRentalAndLeasingIndustry',
    'DP03_0041E': 'WorkersProfessionalScientificManagementAdministrativeAndWasteManagementIndustry',
    'DP03_0042E': 'WorkersEducationalHealthcareAndSocialAssistanceIndustry',
    'DP03_0043E': 'WorkersArtsEntertainmentRecreationAccommodationAndFoodServicesIndustry',
    'DP03_0044E': 'WorkersOtherServicesIndustry',
    'DP03_0045E': 'WorkersPublicAdministrationIndustry',
    'DP03_0047E': 'WorkersPrivateWageAndSalaryClass',
    'DP03_0048E': 'WorkersGovernmentClass',
    'DP03_0049E': 'WorkersSelfEmployedInNotIncorporatedBusinessClass',
    'DP03_0050E': 'WorkersUnpaidFamilyClass',
    'B08301_003E': 'CommutersDriving',
    'B08301_004E': 'CommutersCarpooling',
    'B08301_010E': 'CommutersPublicTransit',
    'B08301_016E': 'CommutersTaxi',
    'B08301_017E': 'CommutersMotorcycle',
    'B08301_018E': 'CommutersBiking',
    'B08301_019E': 'CommutersWalking',
    'B08301_020E': 'CommutersOtherMeans',
    'B08301_021E': 'CommutersWorkingFromHome',
    'B08006_020E': 'MaleCommutersDriving',
    'B08006_021E': 'MaleCommutersCarpooling',
    'B08006_025E': 'MaleCommutersPublicTransit',
    'B08006_031E': 'MaleCommutersBiking',
    'B08006_032E': 'MaleCommutersWalking',
    'B08006_033E': 'MaleCommutersTaxiMotorcycleAndOther',
    'B08006_034E': 'MaleCommutersWorkingFromHome',
    'B08006_037E': 'FemaleCommutersDriving',
    'B08006_038E': 'FemaleCommutersCarpooling',
    'B08006_042E': 'FemaleCommutersPublicTransit',
    'B08006_048E': 'FemaleCommutersBiking',
    'B08006_049E': 'FemaleCommutersWalking',
    'B08006_050E': 'FemaleCommutersTaxiMotorcycleAndOther',
    'B08006_051E': 'FemaleCommutersWorkingFromHome',
    'B08101_010E': 'Age16to19CommutersDriving',
    'B08101_011E': 'Age20to24CommutersDriving',
    'B08101_012E': 'Age25to44CommutersDriving',
    'B08101_013E': 'Age45to54CommutersDriving',
    'B08101_014E': 'Age55to59CommutersDriving',
    'B08101_015E': 'Age60to64CommutersDriving',
    'B08101_016E': 'Age65AndOverCommutersDriving',
    'B08101_018E': 'Age16to19CommutersCarpooling',
    'B08101_019E': 'Age20to24CommutersCarpooling',
    'B08101_020E': 'Age25to44CommutersCarpooling',
    'B08101_021E': 'Age45to54CommutersCarpooling',
    'B08101_022E': 'Age55to59CommutersCarpooling',
    'B08101_023E': 'Age60to64CommutersCarpooling',
    'B08101_024E': 'Age65AndOverCommutersCarpooling',
    'B08101_026E': 'Age16to19CommutersPublicTransit',
    'B08101_027E': 'Age20to24CommutersPublicTransit',
    'B08101_028E': 'Age25to44CommutersPublicTransit',
    'B08101_029E': 'Age45to54CommutersPublicTransit',
    'B08101_030E': 'Age55to59CommutersPublicTransit',
    'B08101_031E': 'Age60to64CommutersPublicTransit',
    'B08101_032E': 'Age65AndOverCommutersPublicTransit',
    'B08101_034E': 'Age16to19CommutersWalking',
    'B08101_035E': 'Age20to24CommutersWalking',
    'B08101_036E': 'Age25to44CommutersWalking',
    'B08101_037E': 'Age45to54CommutersWalking',
    'B08101_038E': 'Age55to59CommutersWalking',
    'B08101_039E': 'Age60to64CommutersWalking',
    'B08101_040E': 'Age65AndOverCommutersWalking',
    'B08101_042E': 'Age16to19CommutersTaxiBikeAndOther',
    'B08101_043E': 'Age20to24CommutersTaxiBikeAndOther',
    'B08101_044E': 'Age25to44CommutersTaxiBikeAndOther',
    'B08101_045E': 'Age45to54CommutersTaxiBikeAndOther',
    'B08101_046E': 'Age55to59CommutersTaxiBikeAndOther',
    'B08101_047E': 'Age60to64CommutersTaxiBikeAndOther',
    'B08101_048E': 'Age65AndOverCommutersTaxiBikeAndOther',
    'B08101_050E': 'Age16to19CommutersWorkingFromHome',
    'B08101_051E': 'Age20to24CommutersWorkingFromHome',
    'B08101_052E': 'Age25to44CommutersWorkingFromHome',
    'B08101_053E': 'Age45to54CommutersWorkingFromHome',
    'B08101_054E': 'Age55to59CommutersWorkingFromHome',
    'B08101_055E': 'Age60to64CommutersWorkingFromHome',
    'B08101_056E': 'Age65AndOverCommutersWorkingFromHome',
    'B08105A_002E': 'WhiteCommutersDriving',
    'B08105A_003E': 'WhiteCommutersCarpooling',
    'B08105A_004E': 'WhiteCommutersPublicTransit',
    'B08105A_005E': 'WhiteCommutersWalking',
    'B08105A_006E': 'WhiteCommutersTaxiBikeAndOther',
    'B08105A_007E': 'WhiteCommutersWorkingFromHome',
    'B08105B_002E': 'BlackCommutersDriving',
    'B08105B_003E': 'BlackCommutersCarpooling',
    'B08105B_004E': 'BlackCommutersPublicTransit',
    'B08105B_005E': 'BlackCommutersWalking',
    'B08105B_006E': 'BlackCommutersTaxiBikeAndOther',
    'B08105B_007E': 'BlackCommutersWorkingFromHome',
    'B08105C_002E': 'IndigenousCommutersDriving',
    'B08105C_003E': 'IndigenousCommutersCarpooling',
    'B08105C_004E': 'IndigenousCommutersPublicTransit',
    'B08105C_005E': 'IndigenousCommutersWalking',
    'B08105C_006E': 'IndigenousCommutersTaxiBikeAndOther',
    'B08105C_007E': 'IndigenousCommutersWorkingFromHome',
    'B08105D_002E': 'AsianCommutersDriving',
    'B08105D_003E': 'AsianCommutersCarpooling',
    'B08105D_004E': 'AsianCommutersPublicTransit',
    'B08105D_005E': 'AsianCommutersWalking',
    'B08105D_006E': 'AsianCommutersTaxiBikeAndOther',
    'B08105D_007E': 'AsianCommutersWorkingFromHome',
    'B08105E_002E': 'PacificIslanderCommutersDriving',
    'B08105E_003E': 'PacificIslanderCommutersCarpooling',
    'B08105E_004E': 'PacificIslanderCommutersPublicTransit',
    'B08105E_005E': 'PacificIslanderCommutersWalking',
    'B08105E_006E': 'PacificIslanderCommutersTaxiBikeAndOther',
    'B08105E_007E': 'PacificIslanderCommutersWorkingFromHome',
    'B08105F_002E': 'OtherRaceCommutersDriving',
    'B08105F_003E': 'OtherRaceCommutersCarpooling',
    'B08105F_004E': 'OtherRaceCommutersPublicTransit',
    'B08105F_005E': 'OtherRaceCommutersWalking',
    'B08105F_006E': 'OtherRaceCommutersTaxiBikeAndOther',
    'B08105F_007E': 'OtherRaceCommutersWorkingFromHome',
    'B08105G_002E': 'TwoOrMoreRacesCommutersDriving',
    'B08105G_003E': 'TwoOrMoreRacesCommutersCarpooling',
    'B08105G_004E': 'TwoOrMoreRacesCommutersPublicTransit',
    'B08105G_005E': 'TwoOrMoreRacesCommutersWalking',
    'B08105G_006E': 'TwoOrMoreRacesCommutersTaxiBikeAndOther',
    'B08105G_007E': 'TwoOrMoreRacesCommutersWorkingFromHome',
    'B08105I_002E': 'HispanicCommutersDriving',
    'B08105I_003E': 'HispanicCommutersCarpooling',
    'B08105I_004E': 'HispanicCommutersPublicTransit',
    'B08105I_005E': 'HispanicCommutersWalking',
    'B08105I_006E': 'HispanicCommutersTaxiBikeAndOther', 
    'B08105I_007E': 'HispanicCommutersWorkingFromHome',
    'C24010A_002E': 'WhiteMaleWorkers',
    'C24010A_003E': 'WhiteMaleWorkersManagementBusinesScienceAndArts',
    'C24010A_004E': 'WhiteMaleWorkersService',
    'C24010A_005E': 'WhiteMaleWorkersSalesAndOffice',
    'C24010A_006E': 'WhiteMaleWorkersResourcesConstructionAndMaintenance',
    'C24010A_007E': 'WhiteMaleWorkersProductionTransportAndMaterialMoving',
    'C24010A_008E': 'WhiteFemaleWorkers',
    'C24010A_009E': 'WhiteFemaleWorkersManagementBusinesScienceAndArts',
    'C24010A_010E': 'WhiteFemaleWorkersService',
    'C24010A_011E': 'WhiteFemaleWorkersSalesAndOffice',
    'C24010A_012E': 'WhiteFemaleWorkersResourcesConstructionAndMaintenance',
    'C24010A_013E': 'WhiteFemaleWorkersProductionTransportAndMaterialMoving',
    'C24010B_002E': 'BlackMaleWorkers',
    'C24010B_003E': 'BlackMaleWorkersManagementBusinesScienceAndArts',
    'C24010B_004E': 'BlackMaleWorkersService',
    'C24010B_005E': 'BlackMaleWorkersSalesAndOffice',
    'C24010B_006E': 'BlackMaleWorkersResourcesConstructionAndMaintenance',
    'C24010B_007E': 'BlackMaleWorkersProductionTransportAndMaterialMoving',
    'C24010B_008E': 'BlackFemaleWorkers',
    'C24010B_009E': 'BlackFemaleWorkersManagementBusinesScienceAndArts',
    'C24010B_010E': 'BlackFemaleWorkersService',
    'C24010B_011E': 'BlackFemaleWorkersSalesAndOffice',
    'C24010B_012E': 'BlackFemaleWorkersResourcesConstructionAndMaintenance',
    'C24010B_013E': 'BlackFemaleWorkersProductionTransportAndMaterialMoving',
    'C24010C_002E': 'IndigenousMaleWorkers',
    'C24010C_003E': 'IndigenousMaleWorkersManagementBusinesScienceAndArts',
    'C24010C_004E': 'IndigenousMaleWorkersService',
    'C24010C_005E': 'IndigenousMaleWorkersSalesAndOffice',
    'C24010C_006E': 'IndigenousMaleWorkersResourcesConstructionAndMaintenance',
    'C24010C_007E': 'IndigenousMaleWorkersProductionTransportAndMaterialMoving',
    'C24010C_008E': 'IndigenousFemaleWorkers',
    'C24010C_009E': 'IndigenousFemaleWorkersManagementBusinesScienceAndArts',
    'C24010C_010E': 'IndigenousFemaleWorkersService',
    'C24010C_011E': 'IndigenousFemaleWorkersSalesAndOffice',
    'C24010C_012E': 'IndigenousFemaleWorkersResourcesConstructionAndMaintenance',
    'C24010C_013E': 'IndigenousFemaleWorkersProductionTransportAndMaterialMoving',
    'C24010D_002E': 'AsianMaleWorkers',
    'C24010D_003E': 'AsianMaleWorkersManagementBusinesScienceAndArts',
    'C24010D_004E': 'AsianMaleWorkersService',
    'C24010D_005E': 'AsianMaleWorkersSalesAndOffice',
    'C24010D_006E': 'AsianMaleWorkersResourcesConstructionAndMaintenance',
    'C24010D_007E': 'AsianMaleWorkersProductionTransportAndMaterialMoving',
    'C24010D_008E': 'AsianFemaleWorkers',
    'C24010D_009E': 'AsianFemaleWorkersManagementBusinesScienceAndArts',
    'C24010D_010E': 'AsianFemaleWorkersService',
    'C24010D_011E': 'AsianFemaleWorkersSalesAndOffice',
    'C24010D_012E': 'AsianFemaleWorkersResourcesConstructionAndMaintenance',
    'C24010D_013E': 'AsianFemaleWorkersProductionTransportAndMaterialMoving',
    'C24010E_002E': 'PacificIslanderMaleWorkers',
    'C24010E_003E': 'PacificIslanderMaleWorkersManagementBusinesScienceAndArts',
    'C24010E_004E': 'PacificIslanderMaleWorkersService',
    'C24010E_005E': 'PacificIslanderMaleWorkersSalesAndOffice',
    'C24010E_006E': 'PacificIslanderMaleWorkersResourcesConstructionAndMaintenance',
    'C24010E_007E': 'PacificIslanderMaleWorkersProductionTransportAndMaterialMoving',
    'C24010E_008E': 'PacificIslanderFemaleWorkers',
    'C24010E_009E': 'PacificIslanderFemaleWorkersManagementBusinesScienceAndArts',
    'C24010E_010E': 'PacificIslanderFemaleWorkersService',
    'C24010E_011E': 'PacificIslanderFemaleWorkersSalesAndOffice',
    'C24010E_012E': 'PacificIslanderFemaleWorkersResourcesConstructionAndMaintenance',
    'C24010E_013E': 'PacificIslanderFemaleWorkersProductionTransportAndMaterialMoving',
    'C24010F_002E': 'OtherRaceMaleWorkers',
    'C24010F_003E': 'OtherRaceMaleWorkersManagementBusinesScienceAndArts',
    'C24010F_004E': 'OtherRaceMaleWorkersService',
    'C24010F_005E': 'OtherRaceMaleWorkersSalesAndOffice',
    'C24010F_006E': 'OtherRaceMaleWorkersResourcesConstructionAndMaintenance',
    'C24010F_007E': 'OtherRaceMaleWorkersProductionTransportAndMaterialMoving',
    'C24010F_008E': 'OtherRaceFemaleWorkers',
    'C24010F_009E': 'OtherRaceFemaleWorkersManagementBusinesScienceAndArts',
    'C24010F_010E': 'OtherRaceFemaleWorkersService',
    'C24010F_011E': 'OtherRaceFemaleWorkersSalesAndOffice',
    'C24010F_012E': 'OtherRaceFemaleWorkersResourcesConstructionAndMaintenance',
    'C24010F_013E': 'OtherRaceFemaleWorkersProductionTransportAndMaterialMoving',
    'C24010G_002E': 'TwoOrMoreRacesMaleWorkers',
    'C24010G_003E': 'TwoOrMoreRacesMaleWorkersManagementBusinesScienceAndArts',
    'C24010G_004E': 'TwoOrMoreRacesMaleWorkersService',
    'C24010G_005E': 'TwoOrMoreRacesMaleWorkersSalesAndOffice',
    'C24010G_006E': 'TwoOrMoreRacesMaleWorkersResourcesConstructionAndMaintenance',
    'C24010G_007E': 'TwoOrMoreRacesMaleWorkersProductionTransportAndMaterialMoving',
    'C24010G_008E': 'TwoOrMoreRacesFemaleWorkers',
    'C24010G_009E': 'TwoOrMoreRacesFemaleWorkersManagementBusinesScienceAndArts',
    'C24010G_010E': 'TwoOrMoreRacesFemaleWorkersService',
    'C24010G_011E': 'TwoOrMoreRacesFemaleWorkersSalesAndOffice',
    'C24010G_012E': 'TwoOrMoreRacesFemaleWorkersResourcesConstructionAndMaintenance',
    'C24010G_013E': 'TwoOrMoreRacesFemaleWorkersProductionTransportAndMaterialMoving',
    'C24010I_002E': 'HispanicMaleWorkers',
    'C24010I_003E': 'HispanicMaleWorkersManagementBusinesScienceAndArts',
    'C24010I_004E': 'HispanicMaleWorkersService',
    'C24010I_005E': 'HispanicMaleWorkersSalesAndOffice',
    'C24010I_006E': 'HispanicMaleWorkersResourcesConstructionAndMaintenance',
    'C24010I_007E': 'HispanicMaleWorkersProductionTransportAndMaterialMoving',
    'C24010I_008E': 'HispanicFemaleWorkers',
    'C24010I_009E': 'HispanicFemaleWorkersManagementBusinesScienceAndArts',
    'C24010I_010E': 'HispanicFemaleWorkersService',
    'C24010I_011E': 'HispanicFemaleWorkersSalesAndOffice',
    'C24010I_012E': 'HispanicFemaleWorkersResourcesConstructionAndMaintenance',
    'C24010I_013E': 'HispanicFemaleWorkersProductionTransportAndMaterialMoving'
}

In [81]:
field_names_by_year_df_renamed = field_names_by_year_df\
    .merge(pd.DataFrame({'FieldName2024':field_rename_dict.keys(), 'TranslatedName':field_rename_dict.values()}),
          on='FieldName2024', how='outer')

In [82]:
field_names_by_year_df_renamed.to_csv('Data/CensusDataFieldsToQuery.csv', index=False)